# 05 Evaluation And Calibration

Compare Elo and logistic regression on probability quality, then calibrate the model with validation predictions.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from sklearn.isotonic import IsotonicRegression

ROOT = Path("..").resolve()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from mlops.data import load_matches
from mlops.evaluation import summarize_probabilities
from mlops.features import build_match_features
from mlops.modeling import elo_probability_from_diff, train_logistic_regression
from mlops.splits import chronological_split, split_features_and_target

DATA_PATH = ROOT / "data" / "matchs_stats.csv"
df = load_matches(DATA_PATH)
feature_df = build_match_features(df)
labeled_df = feature_df[feature_df["blue_team_win"].notna()].copy()
labeled_df["blue_team_win"] = labeled_df["blue_team_win"].astype("Int64")
train_df, valid_df, test_df = chronological_split(labeled_df)
X_train, y_train = split_features_and_target(train_df)
X_valid, y_valid = split_features_and_target(valid_df)
X_test, y_test = split_features_and_target(test_df)

base_model = train_logistic_regression(X_train, y_train)
elo_valid_prob = elo_probability_from_diff(valid_df["elo_diff"])
base_valid_prob = pd.Series(base_model.predict_proba(X_valid)[:, 1], index=valid_df.index)
elo_test_prob = elo_probability_from_diff(test_df["elo_diff"])
base_test_prob = pd.Series(base_model.predict_proba(X_test)[:, 1], index=test_df.index)

In [2]:
pd.DataFrame(
    {
        "elo_valid": summarize_probabilities(y_valid, elo_valid_prob),
        "logreg_valid": summarize_probabilities(y_valid, base_valid_prob),
        "elo_test": summarize_probabilities(y_test, elo_test_prob),
        "logreg_test": summarize_probabilities(y_test, base_test_prob),
    }
)

,elo_valid,logreg_valid,elo_test,logreg_test
log_loss,0.663437,0.688918,0.656142,0.659170
brier_score,0.235268,0.247239,0.232533,0.233682
roc_auc,0.642466,0.624060,0.643344,0.634056
accuracy_50,0.618750,0.575000,0.583851,0.583851


In [3]:
calibrator = IsotonicRegression(out_of_bounds="clip")
calibrator.fit(base_valid_prob, y_valid.astype(int))
calibrated_test_prob = pd.Series(calibrator.predict(base_test_prob), index=test_df.index)

pd.DataFrame(
    {
        "base_logreg_test": summarize_probabilities(y_test, base_test_prob),
        "calibrated_logreg_test": summarize_probabilities(y_test, calibrated_test_prob),
    }
)

,base_logreg_test,calibrated_logreg_test
log_loss,0.659170,0.894863
brier_score,0.233682,0.247979
roc_auc,0.634056,0.616718
accuracy_50,0.583851,0.565217
